# Fenwick & Segment Trees

The structures for **dynamic range queries** — when an array is **updated and queried interleaved**, and you need both fast. Prefix sums (the prefix-sums notebook) answer range queries in O(1) but cost O(n) to update; these give **O(log n)** for *both*. Python ships neither — but a Fenwick tree is ~10 lines, and a segment tree generalizes it to any associative operation.

**Contents**

1. **The problem** — updates *and* queries
2. **Fenwick tree** (Binary Indexed Tree)
3. **Segment tree**
4. **When to use what**
5. **Complexity cheat-sheet**

## 1. The problem — updates *and* queries

A **prefix-sum array** answers "sum of `a[l..r]`" in O(1), but the moment a value changes you must rebuild it — O(n) per update. Fine for a *static* array; it falls apart when updates and queries **interleave** (running balances, leaderboards, "how many values ≤ x seen so far").

These two structures fix that, giving **O(log n)** for *both* update and range query:

- **Fenwick tree (BIT)** — compact and fast, purpose-built for **prefix sums** (and any invertible op you can subtract back out).
- **Segment tree** — bigger but more general: any **associative** operation (sum, min, max, gcd), and extendable to range *updates* with lazy propagation.

## 2. Fenwick tree (Binary Indexed Tree)

A Fenwick tree stores overlapping partial sums in a layout indexed by **the lowest set bit** (`i & -i` — the trick from the bit-manipulation notebook). Index `i` is responsible for a range of length `i & -i` ending at `i`; walking those ranges covers any prefix in O(log n). It's 1-indexed and astonishingly short:

In [1]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)            # 1-indexed

    def update(self, i, delta):              # add delta at position i
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i                      # move to the next responsible node
    def prefix(self, i):                     # sum of [1..i]
        s = 0
        while i > 0:
            s += self.tree[i]
            i -= i & -i                      # strip the lowest set bit
        return s
    def range_sum(self, l, r):               # sum of [l..r], inclusive
        return self.prefix(r) - self.prefix(l - 1)

ft = Fenwick(5)
for i, v in enumerate([1, 2, 3, 4, 5], start=1):
    ft.update(i, v)
print("prefix(3)     :", ft.prefix(3))        # 1+2+3 = 6
print("range_sum(2,4):", ft.range_sum(2, 4))  # 2+3+4 = 9
ft.update(2, 10)                              # a[2] += 10  -> O(log n), no rebuild
print("after a[2]+=10, range_sum(1,5):", ft.range_sum(1, 5))   # 25


prefix(3)     : 6
range_sum(2,4): 9
after a[2]+=10, range_sum(1,5): 25


Feed it **compressed ranks** (coordinate-compression notebook) and a Fenwick tree answers "how many earlier values are < x" over a huge value range — the count-smaller-to-the-right / inversion-count problem that prefix sums can't do dynamically.

## 3. Segment tree

A segment tree stores the array in the **leaves** of a binary tree; each internal node holds the **combine** of its two children. A query walks up from the boundary leaves, merging only the O(log n) nodes that cover the range; a point update fixes one leaf and its log n ancestors. Unlike Fenwick, the combine can be **any associative op** — here sum, but swap `+` for `min` / `max` / `gcd` and nothing else changes:

In [2]:
class SegmentTree:
    def __init__(self, data):
        self.n = len(data)
        self.tree = [0] * (2 * self.n)       # leaves live at indices [n, 2n)
        for i in range(self.n):
            self.tree[self.n + i] = data[i]
        for i in range(self.n - 1, 0, -1):   # build internal nodes bottom-up
            self.tree[i] = self.tree[2 * i] + self.tree[2 * i + 1]

    def update(self, i, value):              # set position i to value
        i += self.n
        self.tree[i] = value
        i //= 2
        while i:
            self.tree[i] = self.tree[2 * i] + self.tree[2 * i + 1]
            i //= 2

    def query(self, l, r):                   # combine over [l, r)  (half-open)
        res = 0
        l += self.n; r += self.n
        while l < r:
            if l & 1: res += self.tree[l]; l += 1
            if r & 1: r -= 1; res += self.tree[r]
            l //= 2; r //= 2
        return res

st = SegmentTree([1, 2, 3, 4, 5])
print("query[1,4) :", st.query(1, 4))         # 2+3+4 = 9
st.update(2, 10)                              # set a[2] = 10
print("after a[2]=10, query[0,5):", st.query(0, 5))   # 22


query[1,4) : 9
after a[2]=10, query[0,5): 22


Swap the `+` combine for `min` / `max` and you get range-minimum / maximum queries; add **lazy propagation** and it handles O(log n) *range* updates (add v to a whole range) — the next step up when you need range-update *and* range-query together.

## 4. When to use what

| You need… | Use | Why |
|---|---|---|
| static array, many range-sum queries | prefix sums | O(1) query, no updates |
| point updates + prefix/range **sums** | Fenwick tree | tiny, fast, O(log n) both |
| point updates + range **min/max/gcd/…** | segment tree | any associative op |
| **range** updates + range queries | segment tree + lazy propagation | both O(log n) |
| count-smaller / inversions over big values | Fenwick over compressed ranks | the classic application |

**The progression:** prefix sums (static) → Fenwick (dynamic sums) → segment tree (dynamic anything) → lazy segment tree (dynamic range updates). Reach only as far up as the problem needs.

## 5. Complexity cheat-sheet

| Structure | Build | Point update | Range query | Op supported |
|---|:---:|:---:|:---:|---|
| Prefix sum | O(n) | O(n) rebuild | O(1) | sum (static) |
| Fenwick tree | O(n log n) | O(log n) | O(log n) | sum / invertible |
| Segment tree | O(n) | O(log n) | O(log n) | any associative |
| Segment tree + lazy | O(n) | O(log n) range | O(log n) | any associative |

<sub>Fenwick is the smallest, fastest choice when you only need sums; a segment tree trades a bigger constant for generality (min/max/gcd) and, with lazy propagation, range updates.</sub>